# Laser PLL: three-stage acquisition and DC interface test

ADC 224/3 → DAC 229/1. Keep the matching `laser_pll.bit`, `laser_pll.hwh`,
`laser_pll.py` and this notebook together. Use the existing board clock initialization
procedure. **Run cells individually**: the DC and generator sections are independent
bench tests, and the later cells engage the laser error output. No cell operates
the analog piezo/EOM switches. The new default polarity is reversed (`invert=True`).
Restart the notebook kernel after replacing `laser_pll.py` so an older imported
helper cannot remain cached. Examples use `ol` for the loaded overlay.


## 1. Load and configure while muted
Loading the overlay reprograms the FPGA. Do not apply the old ADC mixer test setup afterward.


In [ ]:
from pathlib import Path
import sys
import time
from pynq import Overlay

sys.path.insert(0, str(Path.cwd()))
import laser_pll
# Run your normal board-specific reference-clock initialization as usual.
ol = Overlay(str(Path.cwd() / "laser_pll.bit"))
laser_pll.configure(ol, reference_frequency_hz=800e6,
                    phase_gain_shift=0, capture_gain_shift=6,
                    minimum_amplitude=64, invert=True)
print(laser_pll.acquisition_status(ol))


## 2. Independent DC conversion-stage test
Disconnect the error output from the laser controllers for this measurement.
The custom DC interface is already verified; this measures its noise at fixed
codes without ADC/CORDIC/discriminator noise. At a nominal 2 Vpp bipolar range,
code +8192 is approximately +0.25 V before any interface inversion/offset.
The signed code bypasses the PLL polarity/gains and does not need a valid ADC input.


In [ ]:
laser_pll.set_dc_output(ol, 8192)
laser_pll.enable(ol)
time.sleep(0.05)  # Default handoff settles within 4.27 ms.
print(laser_pll.acquisition_status(ol))
# Measure scope noise only when dc_test=True and transitioning=False.


In [ ]:
laser_pll.set_dc_output(ol, 0)
print(laser_pll.acquisition_status(ol))
# Compare zero-code noise. A literal code change in DC mode is immediate.


In [ ]:
laser_pll.set_dc_output(ol, -8192)
print(laser_pll.acquisition_status(ol))
# Compare negative-code noise, then mute in the next cell.


In [ ]:
laser_pll.enable(ol, False)


You can use RFDC DAC mixer settings to translate the constant into an RF tone.
Restore the mixer to zero frequency **and phase** before returning to error output.
`configure()` below does that while muted. No fabric sine generator is included.


## 3. Wide capture with the piezo controller on
The piezo stays on; keep the EOM branch off/reset during coarse acquisition.
Confirm the complete feedback polarity with the existing analog procedure.
The FFT searches 200–950 MHz and updates every 59.03 µs. This error is intended
for slow acquisition, not the MHz EOM loop. The reference DDS remains at 800 MHz.

Use a slow piezo acquisition setting here: a few hundred Hz of unity-gain bandwidth is a conservative starting target if adjustable; determine actual gain/filter settings from the plant. Keep it well below the 16.94 kHz FFT update rate. The processing plus hold delay is roughly 88 µs. Watch the FFT beat approach 800 MHz and check piezo headroom. If it runs away, mute and check feedback polarity.


In [ ]:
laser_pll.configure(ol, reference_frequency_hz=800e6, invert=True)
laser_pll.configure_acquisition(ol, stage=1,
                                transition_interval_log2=4,
                                coarse_gain_shift=0, minimum_fft_peak=32)
laser_pll.enable(ol)
time.sleep(0.05)
print(laser_pll.acquisition_status(ol))


## 4. Enter the existing frequency-assisted capture stage
Run after the piezo has brought the beat near the reference. Hardware waits for
FFT detuning <80 MHz and valid fine phase. The call waits for the handoff ramp
to finish; a timeout restores the previous request. It does not certify lock.

Keep analog settings fixed through this digital transition and wait for `transitioning=False`. Then restore the piezo acquisition settings gradually if necessary. Do not wait for an exactly zero FFT error: bin spacing is 1.92 MHz. Keep EOM off until the remaining frequency excursion fits its actual tuning range.


In [ ]:
laser_pll.select_stage(ol, 2, timeout_s=5.0)
print(laser_pll.status(ol))
print(laser_pll.acquisition_status(ol))


## 5. Enter unwrapped-phase acquisition
First observe that detuning/oscillation fits the intracavity EOM's available throw,
and prepare its analog controller, leaving it off for the initial digital handoff.
The piezo remains on. Stage 3 is a manual handoff: there is no frequency-band,
FFT or dwell requirement. Valid fine-phase data and five clocks of wrap/scaler
preparation are still required. The derivative is removed through an offset ramp;
stage 3 divides phase sensitivity by 160 for the same gain field. At shift 0,
full DAC range covers approximately ±80 turns (1.989 mV/rad at 2 Vpp).
Shifts 1/2/3 give ±40/20/10 turns. The counter still accumulates beyond the rails.
There is no fourth high-gain stage.

The signed 24-bit counter retains turns while the DAC rails. A 1 MHz mismatch for
1 ms creates 1000 turns, requiring equal opposite frequency-time area to unwind.
Expect overshoot; monitor the count and actuator headroom. Settled transition
status does not prove phase lock. Signal gaps clear unobservable cycle history.

For this wide-phase test, leave EOM off while the piezo centers the beat in stage 2. Request stage 3 and wait for the digital handoff to settle. Then engage EOM at conservative FALC gain, keeping the FPGA gain and analog filter switches fixed, and raise FALC gain gradually. The piezo remains on throughout. Stage 3 already excludes the derivative; do not call `enable_capture(False)`. Look for stationary phase/count and actuator headroom. A nonzero stationary count can reflect analog offsets; `fine_ready=True` alone is not proof of lock.

For reacquisition, keep piezo on, turn EOM off/reset, return piezo to its slow setting and request stage 1. If the error is running away, mute before investigating. Do not rely on Python polling to detect every fast excursion.

This behavior requires the rebuilt **A4 wide-phase bitstream** and matching helper.
The helper rejects older A3 firmware before configuration writes. `fine_ready` now means
fine-data validity only. Manual entry does not extend the tested ±100 MHz
fine-detector band or guarantee capture: 10 MHz for 1 ms accumulates 10,000 turns.


In [ ]:
laser_pll.select_stage(ol, 3, timeout_s=5.0)
print(laser_pll.acquisition_status(ol))
print(laser_pll.status(ol))


## 6. Read-only acquisition diagnostic
MMIO polling is irregular and much slower than the detector. It can show slow
frequency excursions and accumulated turns, but cannot resolve every slip or MHz
modulation. Fine frequency is meaningful only inside its unambiguous DDC band.
The coarse trace has 1.92 MHz bins; apparent fine steps are quantization.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

records = []
start_s = time.monotonic()
for _ in range(200):
    fine = laser_pll.status(ol)
    acquisition = laser_pll.acquisition_status(ol)
    records.append((time.monotonic()-start_s,
                    -fine["frequency_error_hz"] if fine["valid"] else np.nan,
                    acquisition["fft_frequency_hz"]-800e6 if acquisition["fft_valid"] else np.nan,
                    acquisition["remembered_turns"]))
    time.sleep(0.002)
time_s, fine_offset_hz, coarse_offset_hz, turns = np.array(records).T
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
axes[0].plot(time_s, fine_offset_hz/1e6, label="Fine frequency difference")
axes[0].plot(time_s, coarse_offset_hz/1e6, label="FFT estimate", alpha=0.6)
axes[0].set_ylabel("Beat − reference (MHz)")
axes[0].legend()
axes[1].plot(time_s, turns)
axes[1].set_ylabel("Remembered error (turns)")
axes[1].set_xlabel("Time (s)")
fig.tight_layout()
plt.show()


## 7. Phase-continuous reference update, if desired
This preserves the DDS accumulator. GPIO timing is software timed. Retuning can
create a real phase/frequency transient even though the reference phase is continuous.


In [ ]:
actual_hz = laser_pll.set_reference_frequency(ol, 800.001e6)
print(f"Reference: {actual_hz:.6f} Hz")


## 8. Mute before changing bench configuration
Mute resets phase/wrap history but the reference DDS continues advancing.


In [ ]:
laser_pll.enable(ol, False)
print(laser_pll.status(ol))


## 9. Independent wrapped-phase test with an 800.1 MHz generator
Disconnect the laser controllers. This explicitly selects legacy stage 0 with
capture disabled, for a direct comparison of arctan phase noise against the DC
test. At an 800 MHz reference and noninverting output interface, the reversed
polarity gives an **upward sawtooth**, period 10 µs. This mode is not the new
unwrapped acquisition stage. Fine capture-on/off comparison is available here.


In [ ]:
laser_pll.configure(ol, reference_frequency_hz=800e6, invert=True)
laser_pll.configure_acquisition(ol, stage=0)
laser_pll.enable(ol)
print(laser_pll.status(ol))


In [ ]:
laser_pll.enable_capture(ol, True)
print(laser_pll.status(ol))
# Observe the extra offset/noise from the derivative. This flag only applies to stage 0.


In [ ]:
laser_pll.enable_capture(ol, False)


## 10. Gain setting example while muted
Gain fields are shifts: 0 means ×1, 4 means ×16. Phase gain affects stages 0, 2
and 3; stage 3 additionally divides the slope by 160. Coarse gain is independent.
At stage 3, shifts 0/1/2/3 give ±80/40/20/10 turns at the rails.
Gain writes are immediate, not crossfaded.
Configure the intended gain before tuning the analog loop.


In [ ]:
laser_pll.enable(ol, False)
laser_pll.set_gains(ol, phase_gain_shift=0, capture_gain_shift=6)
print(laser_pll.status(ol))
